# Agilent BioTek EL406 washer-dispenser quickstart

The EL406 is the whole family in one instrument: a wash manifold that primes, dispenses, aspirates
and washes; a syringe box that meters volumes precisely; and a peristaltic pump that pushes fluid
through a tubing cassette. Which of those it has, and which options go with them — a buffer
switching valve, an ultrasonic cleaner, vacuum filtration — is a matter of what somebody fitted,
and it reports all of it.

This quickstart connects to the instrument, reads what it is and what it has fitted, tells it which
plate is on its carrier, primes and washes through the manifold, dispenses from the syringes and the
pump, explains where in the well a step works, runs a protocol file, and disconnects.

| Property | Value |
|---|---|
| Communication | A serial port, or USB through an FTDI interface |
| Serial parameters | 38400 baud, 8 data bits, 2 stop bits, no parity, no flow control |
| Operations | Washing, priming, dispensing, aspirating, auto-clean, syringe and peristaltic dispensing, shake and soak |
| Plate formats | 96-well, 384-well, 384-well PCR, 1536-well and 1536-flanged |
| Buffer inlets | A, B, C, D — anything but A needs the buffer switching valve |
| Syringes | One box; which syringes a step may drive is decided by the fitted manifold |
| Peristaltic pumps | One, with a tubing cassette |
| Manifold reach | Depth 1-210 motor steps, across the well -60 to 60, along the well -40 to 40 |
| Protocol files | `.LHC` protocol files are read, checked and run |

```{warning}
This driver has not yet been checked against a real EL406. `setup()` says so in the log every time
it runs. Every step it sends has been checked against the vendor's own interface library — frame
for frame, and against the same validation the instrument applies — but that is not the same as
having moved fluid. Verify every protocol on labware and fluid you can afford to lose, keep a hand
on the power switch the first time the carrier moves, and please report what you find on the
[PyLabRobot forum](https://discuss.pylabrobot.org) so the warning can be removed.
```

```{warning}
Follow the manufacturer's installation, fluid-handling and safety instructions. Priming, washing and
dispensing all move fluid: the buffer bottles must be full and the waste bottle empty enough before
anything in this notebook runs.
```

```{device-card} biotek-el406
```

## How it communicates

PyLabRobot frames each command as an 11-byte header and a payload, writes it to the instrument,
reads back an acknowledgement and the reply, and turns a non-zero status into a typed exception.
Which of the two transports carries those bytes is decided by the port string alone, and nothing
above that point knows which one it got.

Operations that move fluid do not answer when they are done. The driver sends them, then polls the
instrument's run state until it stops reporting a step in progress, which is why every method that
touches the instrument is awaited and can take as long as the physical operation does.

Reading `.LHC` protocol files additionally needs `pycryptodome`, which is not a PyLabRobot
dependency: the file format is encrypted, and the cipher is not in the standard library.

In [ ]:
%pip install "pylabrobot[serial]" pycryptodome

# On USB, install the FTDI dependencies instead: "pylabrobot[ftdi]".

## Physical setup and finding the port

Install, plumb and power the instrument according to the manufacturer's instructions. Connect the
buffer bottle to the inlet each step will name, fit a cassette into the peristaltic pump if you
intend to use it, connect the waste bottle, and connect the instrument to the computer.

Then find the port string:

- **Serial.** Pass the operating system's own name for the port: `COM3` on Windows,
  `/dev/ttyUSB0` or `/dev/ttyS4` on Linux and macOS. Anything that is not a USB serial number is
  taken to be a serial port and passed through unexamined.

- **USB.** List the attached FTDI devices and use the reported serial number:

  ```bash
  python -m pylibftdi.examples.list_devices
  ```

  The port is then `ftdi:<serial>`. The form the instrument's own protocol files record,
  `USB EL406 sn:<serial>`, is accepted as well.

Keep the carrier and the area around it clear from here on.

## Turn on logging

The driver reports what it is doing through the standard library's `logging`, and says nothing
otherwise. Without this cell the untested-driver warning below, and every step the instrument runs,
pass silently.

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s: %(message)s")

## See which ports have something behind them

`pyserial` lists every serial port the operating system offers, most of which are kernel
placeholders with no hardware behind them. Skipping those leaves the ports worth trying, and a
USB-serial adapter names the instrument it is wired to.

In [ ]:
from serial.tools.list_ports import comports

candidates = [port for port in comports() if port.description != "n/a" or port.vid is not None]

for port in candidates:
    serial_number = f"  sn:{port.serial_number}" if port.serial_number else ""
    print(f"{port.device:16} {port.description}{serial_number}")

if not candidates:
    print("no port has a device behind it; is the instrument powered and connected?")

## Build the instrument

Constructing the object opens nothing and touches no hardware. It records which port to use, which
model this is, and what to call the instrument in logs and error messages.

In [ ]:
from pylabrobot.agilent.biotek.lhc import EL406

# The port is a serial one unless it carries a device serial number: on USB, pass
# "ftdi:YOUR_SERIAL" instead.
device = EL406(port="COM3", name="EL406")
device

## Connect

`setup()` opens the link, asks whether anything is listening, and reads the options the instrument
has fitted. That read is not optional: every step is encoded against it and every check measures
against it, so a failure here stops the notebook rather than being carried past.

It raises a `BiotekError` if the port will not open, if nothing answers on it, or if the fitted
options cannot be read.

In [ ]:
await device.setup()

## Ask what answered

**The model is declared, not discovered.** The instrument does not report which model it is, so it
is the class you constructed that decides how every step is encoded and which options are read.
`device.settings.family` echoes what was declared, not what is attached.

What the instrument does report is its serial number, and a firmware version record whose part
number says which instrument the installed firmware is built for. An EL406 reports a part number
beginning `718`, and `setup()` has already refused a link whose firmware says otherwise.

In [ ]:
print("serial number:  ", await device.get_serial_number())

version = await device.get_firmware_version()
print("part number:    ", version.part_number)
print("firmware:       ", version.software_version)
print("data version:   ", version.data_version)

## Read the configuration

What `setup()` read is kept as a read-only record of what somebody fitted. It is read-only because
the instrument is: of its whole command vocabulary almost nothing about the configuration can be
written, so a record that could be edited would only mislead.

On this model the record decides more than the plate does: which manifold is fitted and therefore
which plates can be washed, whether buffers other than A can be selected, whether there is an
ultrasonic cleaner to auto-clean with, and whether wells can be filtered under vacuum.

In [ ]:
settings = device.settings

print("family:            ", settings.family.name)
print("wash manifold:     ", settings.washer_manifold.name)
print("valve box:         ", settings.valve_box.name)
print("buffer switching:  ", settings.buffer_switching)
print("vacuum filtration: ", settings.vacuum_filtration)
print("ultrasonic:        ", settings.ultrasonic)
print("cell washing:      ", settings.cell_washing)
print("syringe box:       ", settings.syringe_box.name)
print("syringe manifold:  ", settings.syringe_manifold.name)
print("peri pump:         ", settings.peri_pump)
print("y axis:            ", settings.y_axis_installed)

## Ask what it can run

A model can be built to run a fixed set of operations; a particular instrument runs the subset its
fitted hardware supports. `get_available_steps()` has already applied that, which makes it the
answer to "why was my step refused" before you have written the step.

An EL406 with everything fitted runs twelve step types — the four manifold steps plus auto-clean,
the 1536-well wash, both syringe steps, all three peristaltic ones, and shake/soak.

In [ ]:
for step_type in device.get_available_steps():
    print(step_type.name)

## Tell it which plate is on the carrier

Nothing runs until a plate has been set. Every step carries the height it works at, measured from
the nominal heights of the format on the carrier, so without one there is nothing to measure from
and the driver raises `RejectedError` rather than guessing.

The format is resolved from the PyLabRobot plate resource itself — its columns, its rows, and how
deep its wells are. An EL406 works five: 96-well, 384-well, 384-well PCR, 1536-well and
1536-flanged. The PCR and flanged formats share their shape with an ordinary plate and differ in
something a resource does not carry, so they are only ever named outright.

In [ ]:
from pylabrobot.resources import cor_96_wellplate_360uL_Fb

plate = cor_96_wellplate_360uL_Fb(name="plate")
device.set_plate(plate)

print(device.plate)

## Check before running

`can_run()` measures steps against the instrument as it is now — what is fitted, which plate is on
the carrier, and what the plate will accept — and touches nothing. It is what `run_protocol()` does
first, so calling it yourself is how you see a refusal without moving anything.

The report is truthy when everything can run, and prints as the list of what cannot.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.manifold_prime import ManifoldPrime

report = await device.can_run([ManifoldPrime(volume=10_000, buffer="A")])
print(report)
print("can run:", bool(report))

## Prime the wash manifold

Priming pumps buffer through the manifold until its lines are full and sends it to waste. Nothing is
dispensed into the plate, which makes it the safest operation to try first — but it does move fluid,
so check the bottles before running this cell.

The volume is in microlitres and the instrument meters it at millilitre resolution. The default is
40 mL, a full prime of dry lines; 10 mL is enough to see the pump run.

In [ ]:
await device.washer.prime(volume=10_000, buffer="A", flow_rate=9)

## Dispense, aspirate, and wash

**These dispense into the plate**, so put one on the carrier you are willing to fill. A dispense
fills every selected well; an aspirate empties them; a wash is the two of them repeated, and is one
step rather than a loop — the instrument runs the cycles itself.

The refill a wash performs needs a volume of its own: the step type defaults to no volume at all, so
a wash that names none is refused before anything moves.

In [ ]:
from pylabrobot.agilent.biotek.lhc.protocols.steps.step_parts.positioning import Positioning
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.manifold_dispense import ManifoldDispense

async with device.batch(home_on_close=True):
    await device.washer.dispense(volume=100, buffer="A", flow_rate=7)
    await device.washer.aspirate()
    await device.washer.wash(
        cycles=2,
        dispense=ManifoldDispense(
            volume=100,
            buffer="A",
            positioning=Positioning(z_steps=device.plate.manifold_dispense_height),
        ),
    )

### One buffer inlet per run, unless a valve can switch it

A step names the inlet it draws from — `"A"` through `"D"`. Without the buffer switching valve the
whole protocol is committed to the first inlet a step names, and a later step naming a different one
is refused on the second of the pair, because nothing switches the line mid-run. With the valve
fitted, a protocol may draw from several.

The cell below asks the record first, so it is safe to run on any instrument.

In [ ]:
if device.settings.buffer_switching:
    async with device.batch(home_on_close=True):
        await device.washer.prime(volume=5_000, buffer="A")
        await device.washer.prime(volume=5_000, buffer="B")
else:
    print("no buffer switching valve fitted; one inlet for the whole protocol")

## Auto-clean, and filtering under vacuum

Two options that add operations of their own rather than changing existing ones.

**Auto-clean** soaks the manifold in cleaning fluid and needs the ultrasonic cleaner; its duration
is in seconds. **Vacuum filtration** pulls the wells through a filter plate instead of aspirating
from above, and needs both the module and the filtration carrier — the instrument is asked which
carrier is fitted, and a filtering aspirate is refused if it is the wrong one. Under vacuum the
aspirate's `delay` is a filtration time in seconds rather than a delay in milliseconds.

In [ ]:
if device.settings.ultrasonic:
    await device.washer.auto_clean(duration=600, buffer="A")
else:
    print("no ultrasonic cleaner fitted; nothing to do")

if device.settings.vacuum_filtration:
    await device.washer.aspirate(vacuum_filtration=True, delay=30)
else:
    print("no vacuum filtration fitted; nothing to do")

## The syringes

The syringe box meters a volume precisely, where the manifold delivers what its pump pushes. Volumes
are per well in µL, and each syringe draws from one of two supply bottles named per step.

Which syringes a step may drive is decided by the fitted *manifold*, not by the box: on the plain
16-tube manifold each step drives one syringe. A prime drives one regardless. A syringe is also
committed to one bottle for the whole run unless the syringe-side switching valve is fitted.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.instrument.syringe_box_type import SyringeBoxType

if device.settings.syringe_box is not SyringeBoxType.NOT_INSTALLED:
    async with device.batch(home_on_close=True):
        await device.syringe_dispenser.prime(volume=5_000, syringe="A", syringe_bottle="A1")
        await device.syringe_dispenser.dispense(volume=50, syringe="A", syringe_bottle="A1")
else:
    print("no syringe box fitted; nothing to do")

## The peristaltic pump

One pump, one tubing cassette. Priming fills the tubing; purging empties it, running fluid to waste
rather than into the plate — which is what you want at the end of a run and before a cassette comes
out. A dispense meters a volume per tube; the flow rate is one of three named speeds.

This model has a single pump, so a step naming the secondary one is refused with a reason rather
than quietly running on the primary.

In [ ]:
if device.settings.peri_pump:
    async with device.batch(home_on_close=True):
        await device.peristaltic_dispenser.prime(volume=300, peri_pump="Primary")
        await device.peristaltic_dispenser.dispense(volume=100, peri_pump="Primary")
    await device.peristaltic_dispenser.purge(volume=300, peri_pump="Primary")
else:
    print("no peristaltic pump fitted; nothing to do")

## Washing a 1536-well plate

A 1536-well plate has a wash of its own, and it is a different step: the ordinary wash is refused on
one. The refill comes from the **syringes** rather than the manifold, so its dispense is a
`SyringeDispense` — which means it needs two things fitted, not one: the 128-tube wash manifold, and
one of the 32-tube syringe manifolds, the only ones a 1536-well plate accepts.

Selections work in sectors there rather than columns, which is what lets a manifold with fewer tubes
than the plate has wells cover it in several passes.

In [ ]:
from pylabrobot.agilent.biotek.lhc.enums.instrument.syringe_manifold import SyringeManifold
from pylabrobot.agilent.biotek.lhc.enums.instrument.washer_manifold import WasherManifold
from pylabrobot.agilent.biotek.lhc.protocols.steps.steps.syringe_dispense import SyringeDispense

THIRTY_TWO_TUBE = {SyringeManifold.TUBE_32_LARGE_BORE, SyringeManifold.TUBE_32_SMALL_BORE}

if (
    device.plate.wells == 1536
    and device.settings.washer_manifold is WasherManifold.TUBE_128
    and device.settings.syringe_manifold in THIRTY_TWO_TUBE
):
    await device.washer.wash_1536(cycles=2, dispense=SyringeDispense(volume=5, syringe="A"))
else:
    print("this needs a 1536-well plate, a 128-tube wash manifold and a 32-tube syringe manifold")

## Shake and soak

Shaking and soaking are one step, and either half can be left out by giving it no time: `duration`
shakes, `soak_duration` leaves the plate still afterwards. Both are in seconds. This is the one
operation every instrument in the family offers, whatever is fitted.

In [ ]:
await device.shake(duration=30, intensity="Medium", axis="X", soak_duration=30)

## Where in the well a step works

Every operation that reaches into the plate takes a `positioning`, and defaults it to the nominal
position for that head over the format on the carrier. Three numbers:

- `z_steps` — how deep the head goes. **This is a height, not an offset**: it defaults to the plate
  record's own nominal height for the head, and a larger number reaches further down into the well.
  The manifold works at 1-210 here; the syringes and the pump reach 1-1500.
- `x_steps` — across the well: -60 to 60 for the manifold, ±125 for the syringes and the pump.
- `y_steps` — along the well, ±40 for every head, and it needs the Y axis to be fitted.

The nominal heights come from the plate record, so they change with the plate and differ per head:
the manifold dispenses and aspirates at two different heights, and the syringes at a third.

In [ ]:
print("nominal dispensing height:  ", device.plate.manifold_dispense_height)
print("nominal aspirating height:  ", device.plate.manifold_aspirate_height)
print("nominal syringe height:     ", device.plate.dispenser_height)

```{note}
The offsets are in motor steps rather than millimetres, which is the one place this package deviates
from PyLabRobot's convention — the field names say so. The conversion differs per axis, per model
and per head, and only part of it is established, so the package does not convert. A step is also
what a protocol file stores, which is what lets a protocol be read and written with no instrument to
ask.
```

## Run several operations in one batch

Opening a batch homes the motors, takes the instrument so that nothing else can interleave a run on
it, and holds it until the block ends. Every operation opens one; doing it once around several
operations — as the cells above have been doing — is what stops the motors being homed between each
of them.

`home_on_close=True` drives the transport home before the batch closes. The instrument does not do
this by itself; ask for it when the next thing to touch the plate is a person. Nesting is allowed
and does nothing: an operation called inside an open batch joins it.

In [ ]:
async with device.batch(home_on_close=True):
    await device.washer.prime(volume=5_000, buffer="A")
    await device.washer.dispense(volume=100, buffer="A")
    await device.shake(duration=15, soak_duration=0)

## Watch a step, and stop it

`get_status()` reports what the instrument is doing, which timed phase a running step is in, and how
many seconds are left in it. It can be called at any time, including while a step is running.

An operation does not return until its step has finished, so pausing or aborting means asking from
somewhere else while it runs. In a notebook that is a task. Abort makes the waiting operation raise
`AbortedError`, so a protocol run ends where it was stopped.

In [ ]:
import asyncio

from pylabrobot.agilent.biotek.lhc.error_handling import AbortedError

running = asyncio.create_task(device.washer.prime(volume=40_000, buffer="A"))
await asyncio.sleep(2)

status = await device.get_status()
print("state:    ", status.state.name)
print("activity: ", status.activity.name)
print("remaining:", status.remaining, "s")

await device.abort()
try:
    await running
except AbortedError as error:
    print("stopped:", error)

## Run a protocol file

A `.LHC` protocol file is read into a `Protocol`: what it will run, and everything the file records
alongside it. Reading needs `pycryptodome`, installed at the top of this notebook.

Reading a file never fails on a step it cannot understand — the file's own records are kept as they
are, so a protocol from another model still reads, prints and writes. `build_steps()` is what turns
those records into steps, and it names the one that will not read.

In [ ]:
from pylabrobot.agilent.biotek.lhc import read

protocol = read("path/to/your/protocol.LHC")

print("name:      ", protocol.protocol_name)
print("written by:", protocol.lhc_version)
print("written for:", protocol.instrument_name)
print("plate:     ", protocol.plate_type or protocol.plate_type_number)
print(
    "entries:   ",
    len(protocol.entries),
    "of which",
    len(protocol.device_entries),
    "operate the instrument",
)

for index, step in enumerate(protocol.build_steps()):
    print(f"  step {index}: {type(step).__name__}")

### What the file says about its instrument

A protocol file records the options the instrument had fitted when it was written. Nothing runs
against that record — steps are encoded against the instrument in front of you — and nothing writes
it to the instrument. It is good for exactly one question, worth asking about a file that came from
another machine: was this written for a differently equipped instrument?

`compare_settings()` is truthy when the two agree, and prints as the options that differ. It raises
`ValueError` for a file that carries no such record, which is how the oldest releases wrote one.

In [ ]:
comparison = device.compare_settings(protocol)
print(comparison)
print("same configuration:", bool(comparison))

### Running it

`run_protocol()` checks the protocol, opens one batch around the whole run, sends each step and
polls it to completion. The check is the same `can_run()` from above and happens automatically, so a
protocol that cannot run raises before anything moves.

**This runs whatever the protocol does**, which for most washer protocols means washing every well
of the plate on the carrier. Read the steps printed above first.

The entries that sequence a run rather than operate the instrument — delays, loops, remarks — are
not run; the device steps go in file order. They are still on `protocol.entries` to inspect.

In [ ]:
await device.run_protocol(protocol, home_on_close=True)

Steps built in Python run the same way. `run_protocol()` takes a list of steps as readily
as a protocol, and `run_step()` runs a single one.

In [ ]:
await device.run_protocol(
    [
        ManifoldPrime(volume=10_000, buffer="A"),
        ManifoldDispense(
            volume=100,
            buffer="A",
            positioning=Positioning(z_steps=device.plate.manifold_dispense_height),
        ),
    ],
    home_on_close=True,
)

## Home the transport and disconnect

Homing drives the transport to its home position and confirms it arrived. Do it before a person
reaches for the plate, unless the last batch already closed with `home_on_close=True`.

`stop()` closes the link. It does nothing on an instrument that is already closed, so it is safe to
run this cell twice, and it is worth running from a `finally` in a script so that a failed run does
not leave the port open.

In [ ]:
await device.home()
await device.stop()

```{note}
The fluid left in the manifold and the cassette after a run is the instrument's problem, not the
driver's. Follow the manufacturer's shutdown and maintenance procedure — `auto_clean(...)` soaks the
manifold, `peristaltic_dispenser.purge(...)` empties a cassette before it comes out, and most
maintenance routines ship as protocol files you can run with `run_protocol()`.
```